# GPT-2 pretraining Fine-tuning Notebook

This notebook demonstrates how to fine-tune a GPT-2 model on custom text data.

In [26]:
# Import required libraries
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch
import tqdm
import numpy as np
import time
import re
from pathlib import Path

## Custom Dataset Class

This class handles the data preprocessing and chunking of our text data.

In [33]:
class MyDataset(Dataset):
    def __init__(self, file_path, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        # Read and clean the text file
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()
        
        # Tokenize the entire text
        encodings = tokenizer(text, truncation=False, return_tensors='pt')
        self.input_ids = encodings.input_ids[0]
        
        # Create chunks of max_length
        self.chunks = []
        for i in range(0, len(self.input_ids) - max_length + 1, max_length // 2):
            chunk = self.input_ids[i:i + max_length]
            if len(chunk) == max_length:
                self.chunks.append(chunk)

        # Print information about chunks
        print(f"\nTotal number of chunks: {len(self.chunks)}")
        
        print("\nFirst chunk:")
        print(tokenizer.decode(self.chunks[0]))
        print("\nLast chunk:")
        print(tokenizer.decode(self.chunks[-1]))
        print("-" * 50)
    
    def __len__(self):
        return len(self.chunks)
    
    def __getitem__(self, idx):
        chunk = self.chunks[idx]
        attention_mask = torch.ones_like(chunk)
        return chunk, attention_mask

## Text Generation Function

In [28]:
def generate_text(model, tokenizer, prompt, device, max_length=30):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    
    output = model.generate(
        input_ids,
        max_length=max_length,
        pad_token_id=tokenizer.pad_token_id,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        top_k=50,
        num_return_sequences=1
    )
    
    return tokenizer.decode(output[0], skip_special_tokens=True)

## Training Function

This function handles the training loop for our model.

In [29]:
def train(dataloader, model, optimizer, device):
    model.train() # set model to train mode
    epochs = 5
    losses = []
    
    # Add timing
    start_time = time.time()
    
    # Print first 5 batches before training starts
    print("\nFirst 2 batches:")
    for i, (batch_ids, attention_mask) in enumerate(dataloader):
        if i >= 2:
            break
        print(f"\nBatch {i + 1}:")
        print("-" * 50)
        print(f"Batch IDs shape: {batch_ids.shape}")
        print(f"Attention mask shape: {attention_mask.shape}")
        print("Batch text:")
        for i, batch in enumerate(batch_ids):
            print(f"\nSequence {i + 1}:")
            print(f"Text: {tokenizer.decode(batch)}")
            print(f"Attention mask: {attention_mask[i]}")
            print("-" * 20)  # separator between batch items
    
    for epoch in tqdm.tqdm(range(epochs)):
        epoch_start_time = time.time()
        epoch_losses = []
        print(f"\nEpoch {epoch + 1}/{epochs}")
        
        for batch_ids, attention_mask in dataloader:
            # 1. Move data to correct device (GPU/CPU)
            batch_ids = batch_ids.to(device)
            attention_mask = attention_mask.to(device)
            
            # 2. Clear previous gradients
            optimizer.zero_grad()
            
            # 3. Forward pass: model makes predictions
            outputs = model(batch_ids, attention_mask=attention_mask, labels=batch_ids)
            loss = outputs.loss
            
            # 4. Backward pass: compute gradients
            loss.backward()
            
            # 5. Update weights
            optimizer.step()
            
            epoch_losses.append(loss.item())
        
        avg_loss = sum(epoch_losses) / len(epoch_losses)
        losses.append(avg_loss)
        
        # Calculate and print timing info
        epoch_time = time.time() - epoch_start_time
        total_time = time.time() - start_time
        print(f"Average loss: {avg_loss:.4f}")
        print(f"Epoch time: {epoch_time:.2f} seconds")
        print(f"Total training time so far: {total_time:.2f} seconds")
        
        # Save losses and model checkpoint
        np.savetxt('losses/pretrain_losses.txt', losses)
        torch.save(model.state_dict(), f"finetuned_model/pretrain_model_state_epoch.pt")
        
        # Generate sample text
        model.eval() # set model to eval mode
        sample_text = generate_text(model, tokenizer, "How to configure power saving on IP phones", device)
        print("\nSample generation:")
        print(sample_text)
        print("-" * 50)
        model.train() # set model to train mode

    total_training_time = time.time() - start_time
    print(f"\nTotal training completed in {total_training_time:.2f} seconds")

## Setup and Training

Initialize the model, tokenizer, and start training

In [30]:
# Setup device
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

# Initialize tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained("gpt2")
model = model.to(device)


Using device: cpu


## Test Model Before Training

In [31]:
print("\nTesting the model BEFORE training:")
model.eval()
test_prompts = [
    "List the audio codecs supported by 9800 series",
    "What are the WiFi standards supported?",
    "What languages are supported on 9800 series?",
    "List the Bluetooth capabilities of 9800 series"
]

for prompt in test_prompts:
    generated = generate_text(model, tokenizer, prompt, device)
    print(f"\nPrompt: {prompt}")
    print(f"Generated: {generated}")
    print("-" * 50)


Testing the model BEFORE training:

Prompt: List the audio codecs supported by 9800 series
Generated: List the audio codecs supported by 9800 series devices.

The following table lists the supported audio codecs, and the number of supported audio
--------------------------------------------------

Prompt: What are the WiFi standards supported?
Generated: What are the WiFi standards supported?

WiFi standards are defined as the basic and most basic standard for wireless communication. The FCC defines WiFi as
--------------------------------------------------

Prompt: What languages are supported on 9800 series?
Generated: What languages are supported on 9800 series?

Windows XP, Vista, 7, 8, 9, 10, 11, 12, 13,
--------------------------------------------------

Prompt: List the Bluetooth capabilities of 9800 series
Generated: List the Bluetooth capabilities of 9800 series Bluetooth® and wireless headsets and their compatibility with Windows Phone 8.1 and Windows Phone 8.1 Mobile.
-----

### Load your data

In [34]:
# Load your data
markdown_dir = "data/9800-datasheet.txt"
dataset = MyDataset(markdown_dir, tokenizer)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)


Total number of chunks: 43

First chunk:
Desk Phone 9800 Series Data Sheet - Cisco Product overview\n\nProduct description\n\nCisco is bringing a new standard to desk phones with a portfolio built for working in modern office environments. The Desk Phone 9800 Series was designed with IT and facilities in mind to deliver 4 new phone models. Included on each phone is the newly released PhoneOS software that simplifies the user experience and compliments our video portfolios RoomOS devices to give a seamless experience from desk spaces to meeting rooms. With expanded functionality \u2013 the 9800 Series combines secure enterprise calling, meetings, desk reservations, sustainability, and emergency notifications all in one device. You don\u2019t need to buy a dedicated device for each feature \u2013 all the features are built into each phone.\n\n[![A close-up of a telephoneDescription automatically generated](/c/dam/en/us/products/collateral/collaboration-endpoints/ip-phones/desk-phone-980

## Train the Model

In [35]:
# Initialize optimizer
optimizer = Adam(model.parameters(), lr=5e-5)

# Train the model
print("\nStarting pretraining...")
train(dataloader, model, optimizer, device)


Starting pretraining...

First 2 batches:

Batch 1:
--------------------------------------------------
Batch IDs shape: torch.Size([8, 512])
Attention mask shape: torch.Size([8, 512])
Batch text:

Sequence 1:
Text:  most such as emergency alerts, security, IT, front desk, broadcast/paging, and so many more. |\n| **Enhanced audio features including noise removal** | \u25cf\u00a0 AI noise removal for both send and receive sides.<br><br>\u25cf\u00a0 Increased audio quality with AI by converting narrowband audio to wideband audio.<br><br>\u25cf\u00a0 Built-in speaker is enhanced by a larger speaker driver and new bass-enhancing acoustic enclosure for more real, natural-sounding voices. |\n| **One button to join a meeting** | \u25cf\u00a0 Experience One Button to Push (OBTP) to quickly join any meeting from the device\u2019s screen (applicable when registered to Webex Calling). |\n| **Security** | \u25cf\u00a0 Security is core to the design, equipped with government-grade security, so comm

  0%|          | 0/5 [00:00<?, ?it/s]


Epoch 1/5


  0%|          | 0/5 [02:56<?, ?it/s]

Average loss: 1.9817
Epoch time: 176.47 seconds
Total training time so far: 176.55 seconds


FileNotFoundError: [Errno 2] No such file or directory: 'losses/pretrain_losses.txt'

## Test Model After Training

In [17]:
# Count total parameters
total_params = sum(p.numel() for p in model.parameters())

# Count trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# Print detailed info
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}\n")

# Optional: See parameter count by layer
for i, (name, param) in enumerate(model.named_parameters()):
    if i >= 5:
        break
    print(f"{name}: {param.numel():,} parameters")

Total parameters: 124,439,808
Trainable parameters: 124,439,808

transformer.wte.weight: 38,597,376 parameters
transformer.wpe.weight: 786,432 parameters
transformer.h.0.ln_1.weight: 768 parameters
transformer.h.0.ln_1.bias: 768 parameters
transformer.h.0.attn.c_attn.weight: 1,769,472 parameters


In [14]:
print("\nTesting the model AFTER training:")
model.eval()
for prompt in test_prompts:
    generated = generate_text(model, tokenizer, prompt, device)
    print(f"\nPrompt: {prompt}")
    print(f"Generated: {generated}")
    print("-" * 50)


Testing the model AFTER training:

Prompt: How to configure power saving on IP phones
Generated: How to configure power saving on IP phones.

This guide explains how to configure the power saving feature on the phone.

Configure power
--------------------------------------------------

Prompt: Steps to set up emergency calling on IP phones
Generated: Steps to set up emergency calling on IP phones can be found in the following section.

IP Phone Configuration Configuration

Before you begin,
--------------------------------------------------

Prompt: Steps to set up call forwarding
Generated: Steps to set up call forwarding for your voicemail service.

Set up a route for voicemail messages to the specified destination.


--------------------------------------------------

Prompt: Deep sleep mode configuration steps
Generated: Deep sleep mode configuration steps to enable or disable the user to sleep using the Sleep Mode Settings feature. This feature is available only to users who enab